# SOP-UQ tester notebook

Run the cells from top to bottom. You will:

1. connect to your team's examiner and to the Qwen model service;
2. look at the data and call business tools and the oracle by hand;
3. run the baseline agent, collect labeled trajectories and fit risk estimators;
4. run the method on the 360 test cases, check `results.csv` and submit it.

You need four values from the examiner: examiner URL, team token, model URL and
model token. Read the problem statement `docs/problem_statement.pdf` and
`docs/quickstart.pdf` first.

In [ ]:
from pathlib import Path
import json, os, sys

# Works whether Jupyter starts in the project folder or in scripts/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "test_client").is_dir())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
WORK = ROOT / ".runs" / "notebook"          # all outputs of this notebook (git-ignored)

# Your examiner and model connections. Set them as environment variables before
# starting Jupyter, or type them here (do not share notebooks containing tokens).
EXAMINER_URL = os.environ.get("SOP_EXAMINER_URL", "http://127.0.0.1:8801")
EXAMINER_TOKEN = os.environ.get("SOP_EXAMINER_TOKEN", "")
MODEL_URL = os.environ.get("SOP_MODEL_URL", "http://127.0.0.1:18080")
MODEL_TOKEN = os.environ.get("SOP_MODEL_TOKEN", "")

## 1. Connect

`spec()` shows the costs, the per-case limits, which of the 360 test cases are
interactive (tools and oracle allowed), and how many scored tries your team has left.

In [ ]:
from test_client import ExaminerClient, ModelClient, RemoteError

examiner = ExaminerClient(EXAMINER_URL, EXAMINER_TOKEN)
spec = examiner.spec()
print("costs:", spec["costs"], "| budget per case:", spec["budget"],
      "| tool limit:", spec["tool_limit"], "| oracle limit:", spec["oracle_limit"])
print("test cases:", len(spec["test_ids"]), "| interactive:", len(spec["interactive_ids"]),
      "| frozen initial answers:", spec["frozen_initials"])
print("tries left:", spec["submissions_left"], "of", spec["max_submissions"],
      "| best try is decided by", spec["primary_metric"])

model = ModelClient(MODEL_URL, MODEL_TOKEN)
print("model:", model.model_id)

## 2. The data

Labeled cases are in `data/train/` (splits `train`, `cal_fit`, `cal_select`,
`dev`); the test inputs are in `data/test/cases.jsonl`, and the examiner's frozen
initial answers for them in `data/test/initial_M9.jsonl`.

In [ ]:
from collections import Counter
from baselines.runner import LABELED_SPLITS, labeled_cases, load_cases, load_frozen

labeled = load_cases(DATA)
test_cases = load_cases(DATA, ["test"])
frozen = load_frozen(DATA)
print(Counter((c["training_split"], c["domain"]) for c in labeled))
print("test cases:", len(test_cases), "| frozen initial answers:", len(frozen))

example = {d: next(c for c in labeled if c["domain"] == d and c["training_split"] == "train")
           for d in spec["domains"]}
for domain, case in example.items():
    print(f"\n{domain}: inputs {sorted(case['inputs'])}\n  label {case['label']}")

## 3. Tools and the oracle by hand

An *episode* is one attempt at one case. Labeled cases get a fresh episode each
time you open them; each test case has one episode per team that keeps all calls
and costs. A tool call costs 0.01 (0.03 with `mode="independent"` on noisy tools);
an oracle read returns the true value of one field for 0.10, and asking again for
the same field is free.

In [ ]:
def input_tool(episode, name, mode="default"):
    """Call a tool whose required arguments all come from the case inputs."""
    tool = next(t for t in episode["view"]["tools"] if t["name"] == name)
    required = tool["inputSchema"]["json"].get("required", [])
    args = {k: episode["view"]["inputs"][k] for k in required}
    return examiner.tool(episode["episode_id"], name, args, mode)["receipt"]

dg = examiner.episode(example["dangerous_goods"]["case_id"])
print("tools:", [t["name"] for t in dg["view"]["tools"]])
print(input_tool(dg, "calculate_sds_label_score"))
print(input_tool(dg, "calculate_sds_label_score", mode="independent"))   # second source
print(examiner.oracle(dg["episode_id"], "sds", "assessment")["receipt"])   # the truth
history = examiner.history(dg["episode_id"])
print("cost so far:", history["cost"], "| tool calls:", history["tool_calls"],
      "| oracle calls:", history["oracle_calls"])

In [ ]:
air = examiner.episode(example["aircraft_inspection"]["case_id"])
print(input_tool(air, "VerifyAircraftClearance"))

cs = examiner.episode(example["customer_service"]["case_id"])
print(input_tool(cs, "validateAccount"))
try:   # customer KPIs exist only after diagnostics, so this is refused
    examiner.oracle(cs["episode_id"], "latency", "pre_repair")
except RemoteError as error:
    print("refused:", error)

## 4. One baseline episode with Qwen

The agent reads the SOP, calls tools one at a time and ends with a
`final_proposal` that includes a self-reported error probability. Four extra
tool-free samples from the same evidence give the `disagreement` risk.

In [ ]:
from baselines.agent import INITIAL_CALL_LIMIT, main_loop, new_snapshot, shared_samples
from baselines.uq import disagreement

case = example["dangerous_goods"]
snapshot = new_snapshot(examiner.episode(case["case_id"]), model)
main_loop(snapshot, model, examiner, limit=INITIAL_CALL_LIMIT)
shared_samples(snapshot, model)
print("answer:", snapshot["answer"], "| label:", case["label"])
print("self-reported risk:", snapshot["p_error"], "| disagreement:", disagreement(snapshot))
tools = [e["receipt"]["tool"] for e in snapshot["history"] if e["kind"] == "tool_receipt"]
print("tool calls:", tools, "| cost:", snapshot["cost"])

## 5. Collect labeled trajectories and fit risk estimators

One pass collects everything: `run_cases` with the `fixed` policy runs the agent
on each labeled case (initial answer and four samples), reads one oracle fact
and lets the agent continue. `initial.jsonl` then trains the initial risk and
`posterior.jsonl` the two post-query risks. Progress is saved after every step,
so re-running the cell resumes.

`LIMIT` cases per split keeps this first pass short. A labeled case takes about
10 seconds on one GPU on average (customer-service cases about 15 seconds), so
all 660 cases (`LIMIT = None`) take about 2 hours; start that early. `fit` uses
`train` for the historical and supervised estimators, `cal_fit` for calibration
and `cal_select` for the DEFER thresholds.

In [ ]:
from baselines.runner import run_cases
from baselines.storage import read_jsonl
from baselines.uq import Calibrator

LIMIT = 4   # cases per split for a first pass; None = all 660
cases = labeled_cases(DATA, LABELED_SPLITS, LIMIT)
print(run_cases(examiner, cases, WORK / "collect", model=model, policy="fixed", fixed_count=1))

initial = read_jsonl(WORK / "collect" / "initial.jsonl")
posterior = read_jsonl(WORK / "collect" / "posterior.jsonl")
calibrator = Calibrator().fit(labeled, initial)
calibrator.select_thresholds(labeled, initial, split="cal_select")
calibrator.fit_posterior(labeled, posterior)
try:
    calibrator.select_thresholds(labeled, posterior, split="cal_select", posterior=True)
except ValueError:
    print("No queried cal_select case yet; the initial thresholds apply after queries.")
calibrator.save(WORK / "calibrator.json")
print(calibrator.fit_counts)
print({method: round(record["threshold"], 3) for method, record in calibrator.selections.items()})

### The two post-query risks

After an oracle read the agent continues. Its new self-reports estimate
`p_initial_error_after_queries` (is the *initial* answer wrong?) and
`p_final_error` (is the *final* answer wrong?); `fit_posterior` above calibrated
both on the labeled query trajectories. The test run below must use the same
query policy, so `METHOD` keeps it in one place.

In [ ]:
METHOD = dict(uq="calibrated_disagreement", policy="fixed", fixed_count=1)

### Score labeled results locally

Every run writes `results.csv`. On labeled cases you can score it yourself with
the same rules the examiner uses; this costs no tries.

In [ ]:
from test_client import read_results, score
from test_client.results import check_results

def local_score(path):
    index = {c["case_id"]: c for c in labeled}
    rows = check_results(read_results(path), index, set(index), require_all=False)
    return score(rows, {cid: index[cid] for cid in rows}, set(index))

report = local_score(WORK / "collect" / "results.csv")
for section in ("initial", "updated_initial", "final"):
    print(section, report[section]["domain_macro"])
print("decision", report["decision"]["domain_macro"])

## 6. Run on the test cases

The examiner fixed one initial answer per test case (same for every team). The
runner estimates its risk on all 360 cases; on the 72 interactive cases it also
queries, lets the agent continue and decides `AUTO` or `DEFER`. Non-interactive
cases need no model calls. Oracle reads and tool calls on test cases are billed
to your team and stay in its ledger.

In [ ]:
test_run = run_cases(examiner, test_cases, WORK / "test", model=model, frozen=frozen,
                     calibrator=calibrator, **METHOD)
print(test_run)
print((WORK / "test" / "results.csv").read_text().splitlines()[:3])

## 7. Submit

Each valid upload uses one of your tries and is scored immediately. A file that
fails the format check is rejected, and re-uploading an identical file returns
its earlier report; neither uses a try. Your best try (lowest primary metric)
counts. Use the feedback to judge your method, not to infer test labels.
`scripts/run_test.py` does the same from a terminal.

In [ ]:
results = WORK / "test" / "results.csv"
test_index = {c["case_id"]: c for c in test_cases}
check_results(read_results(results), test_index, set(spec["interactive_ids"]))

answer = examiner.submit(results)          # uses one try
print("try", answer["submission"], "| tries left:", answer["submissions_left"],
      "| best:", answer["best"])
for section in ("initial", "updated_initial", "final", "decision"):
    print(section, answer["report"][section]["domain_macro"])

In [ ]:
for item in examiner.submissions()["submissions"]:
    print("try", item["submission"], item["report"]["primary"])

## 8. Your own method

Any method works as long as it writes the same `results.csv`: one row per test
case with `p_initial_error`; on interactive cases also `mode`, the final answer,
`p_initial_error_after_queries` and `p_final_error`. The initial answer must be
the frozen one. Example: shrink the baseline risks toward each domain's
historical error rate, then check the file before submitting.

In [ ]:
from test_client import write_results

rows = read_results(WORK / "test" / "results.csv")
domain_of = {c["case_id"]: c["domain"] for c in test_cases}
for case_id, row in rows.items():
    prior = calibrator.rates[domain_of[case_id]]
    if row["p_initial_error"] is not None:
        row["p_initial_error"] = 0.7 * row["p_initial_error"] + 0.3 * prior
mine = write_results(WORK / "my_results.csv", [{"case_id": k, **v} for k, v in rows.items()])
check_results(read_results(mine), test_index, set(spec["interactive_ids"]))
print("ready to submit:", mine)